### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\venka\AppData\Local\Temp\ipykernel_36208\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\venka\Downloads\langchain_upgraded\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../rag_data")

Found 2 PDF files to process

Processing: form 16.pdf
  ✓ Loaded 2 pages

Processing: Sandeep_Ramireddy_Resume (1).pdf
  ✓ Loaded 2 pages

Total documents loaded: 4


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'JasperReports (Form-16)', 'creationdate': '2026-06-03T12:44:33+05:30', 'moddate': '2026-06-03T12:44:33+05:30', 'source': '..\\rag_data\\pdf\\form 16.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'form 16.pdf', 'file_type': 'pdf'}, page_content='FORM NO. 16\n[See rule 31(1)(a)]\nCertificate under Section 203 of the Income-tax Act, 1961 for tax deducted at source on salary paid to an employee under section 192 or pension/interest income\nof specified senior citizen under section 194P\nName and address of the Employer/Specified Bank\nTIGER ANALYTICS INDIA CONSULTING PRIVATE LIMITED\nNO 143, RMZMILLENIA BUSINESS PARK, MGR ROAD,\nKANDANCHAVADI, PERUNGUDI, CHENNAI - 600096\nTamil Nadu\nName and address of the Employee/Specified senior citizen\nVENKATA SIVA SANDEEP RAMIREDDY\n7-150, IPPANAPADU, RAMIREDDY STREET, MANDAPETA\nMANDAL, EASTGODAVARI - 533340 Andhra Pradesh\nPAN of the Deductor\nAAICT8986D\nTA

In [3]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [4]:
chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 15 chunks

Example chunk:
Content: FORM NO. 16
[See rule 31(1)(a)]
Certificate under Section 203 of the Income-tax Act, 1961 for tax deducted at source on salary paid to an employee under section 192 or pension/interest income
of speci...
Metadata: {'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'JasperReports (Form-16)', 'creationdate': '2026-06-03T12:44:33+05:30', 'moddate': '2026-06-03T12:44:33+05:30', 'source': '..\\rag_data\\pdf\\form 16.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'form 16.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'JasperReports (Form-16)', 'creationdate': '2026-06-03T12:44:33+05:30', 'moddate': '2026-06-03T12:44:33+05:30', 'source': '..\\rag_data\\pdf\\form 16.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'form 16.pdf', 'file_type': 'pdf'}, page_content='FORM NO. 16\n[See rule 31(1)(a)]\nCertificate under Section 203 of the Income-tax Act, 1961 for tax deducted at source on salary paid to an employee under section 192 or pension/interest income\nof specified senior citizen under section 194P\nName and address of the Employer/Specified Bank\nTIGER ANALYTICS INDIA CONSULTING PRIVATE LIMITED\nNO 143, RMZMILLENIA BUSINESS PARK, MGR ROAD,\nKANDANCHAVADI, PERUNGUDI, CHENNAI - 600096\nTamil Nadu\nName and address of the Employee/Specified senior citizen\nVENKATA SIVA SANDEEP RAMIREDDY\n7-150, IPPANAPADU, RAMIREDDY STREET, MANDAPETA\nMANDAL, EASTGODAVARI - 533340 Andhra Pradesh\nPAN of the Deductor\nAAICT8986D\nTA

### embedding And vectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9352.15it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\venka\AppData\Local\Temp\ipykernel_36208\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../rag_data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                # metadata={"description": "PDF document embeddings for RAG"}
                 metadata={
                            "description": "PDF document embeddings for RAG",
                            "hnsw:space": "cosine"   # ← ADD THIS
                        }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [8]:
chunks

[Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'JasperReports (Form-16)', 'creationdate': '2026-06-03T12:44:33+05:30', 'moddate': '2026-06-03T12:44:33+05:30', 'source': '..\\rag_data\\pdf\\form 16.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'form 16.pdf', 'file_type': 'pdf'}, page_content='FORM NO. 16\n[See rule 31(1)(a)]\nCertificate under Section 203 of the Income-tax Act, 1961 for tax deducted at source on salary paid to an employee under section 192 or pension/interest income\nof specified senior citizen under section 194P\nName and address of the Employer/Specified Bank\nTIGER ANALYTICS INDIA CONSULTING PRIVATE LIMITED\nNO 143, RMZMILLENIA BUSINESS PARK, MGR ROAD,\nKANDANCHAVADI, PERUNGUDI, CHENNAI - 600096\nTamil Nadu\nName and address of the Employee/Specified senior citizen\nVENKATA SIVA SANDEEP RAMIREDDY\n7-150, IPPANAPADU, RAMIREDDY STREET, MANDAPETA\nMANDAL, EASTGODAVARI - 533340 Andhra Pradesh\nPAN of the Deductor\nAAICT8986D\nTA

In [9]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 15 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]

Generated embeddings with shape: (15, 384)
Adding 15 documents to vector store...
Successfully added 15 documents to vector store
Total documents in collection: 15


### Retriever Pipeline From VectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [11]:
rag_retriever

In [12]:
rag_retriever.retrieve("Professional summary")

Retrieving documents for query: 'Professional summary'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9e27d69a_1',
  'content': 'Assessment Year\n2026-27\nCIT (TDS)\nThe Commissioner of Income Tax (TDS)\n7th Floor, New Block, Aayakar Bhawan, 121 , M.G. Road,\nChennai - 600034\nPeriod with the Employer\nTo\n31-Mar-2026\nFrom\n01-Apr-2025\nSummary of amount paid/credited and tax deducted at source thereon in respect of the employee\nQuarter(s)\nReceipt Numbers of original\nquarterly statements of TDS\nunder sub-section (3) of\nSection 200\nAmount of tax deducted\n(Rs.)\nAmount of tax deposited / remitted\n(Rs.)Amount paid/credited\nQ1 QWAMAASF 72534.00 72534.00597726.00\nQ2 QWCGNQRB 99361.00 99361.00697726.00\nQ3 QWEFQWSF 72534.00 72534.00597726.00\nQ4 QWGWTVLF 89317.00 89317.00651522.00\nTotal (Rs.) 333746.00 333746.002544700.00\nSl. No.\nTax Deposited in respect of the\ndeductee\n(Rs.)\nBook Identification Number (BIN)\nReceipt Numbers of Form\nNo. 24G\nDDO serial number in Form no.\n24G\nDate of transfer voucher\n(dd/mm/yyyy)\nStatus of matching\nwith Form no. 24G\nTotal 

In [13]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 89.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f7983b6d_8',
  'content': 'Ramireddy Venkata Siva Sandeep \nvenkatasivasandeep@gmail.com  \n8727833268  \nHyderabad, Telangana \nProfessional Summary \n \n\uf0b7 Machine Learning professional with eight plus years of experience in the Data and AI sector, proficient in \nmanaging the full lifecycle of data science projects, including exploratory data analysis, feature engineering, \nmodel development and deployment.  \n\uf0b7 Skilled in building multi-agent GenAI systems using Google ADK, enabling intelligent orchestration, automated \nworkflows, and scalable data integration across enterprise platforms. \n\uf0b7 Experienced in designing, implementing, and iterating on generative AI models such as GPT, Gemini models with \na focus on Retrieval-Augmented Generation to improve model responses and data retrieval.  \n\uf0b7 Proficient in optimizing models for coherence, accuracy, minimal hallucination, and maximum coverage. \n\uf0b7 Successfully migrated Oozie models to GCP Clo

### RAG Pipeline- VectorDB To LLM Output Generation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

True

In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.messages import SystemMessage,HumanMessage,AIMessage

In [23]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3-32b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [24]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: qwen/qwen3-32b
Groq LLM initialized successfully!


In [33]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("professional summary of my resume of sandeep ramireddy?")

Retrieving documents for query: 'professional summary of my resume of sandeep ramireddy?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ac962d83_5',
  'content': "CHENNAI, CHENNAI\n03-Jun-2026\nFINANCE HEADDesignation: Full Name:VENU GOPALA RAO  KODE\nNotes:\n1. Part B (Annexure) of the certificate in Form No.16 shall be issued by the employer.\n2. If an assessee is employed under one employer during the year, Part 'A' of the certificate in Form No.16 issued for the quarter ending on 31st March of the financial year shall contain the details\nof tax deducted and deposited for all the quarters of the financial year.\n3. If an assessee is employed under more than one employer during the year, each of the employers shall issue Part A of the certificate in Form No.16 pertaining to the period for which such\nassessee was employed with each of the employers. Part B (Annexure) of the certificate in Form No. 16 may be issued by each of the employers or the last employer at the option of the assessee.\n4. To update PAN details in Income Tax Department database, apply for 'PAN change request' through NSDL or UTITSL.

### Integration Vectordb Context pipeline With LLM output

In [34]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3-32b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [35]:
answer=rag_simple("professional summary of my resume of sandeep ramireddy?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'professional summary of my resume of sandeep ramireddy?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 112.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


<think>
Okay, let's see. The user is asking for a professional summary of their resume, specifically for someone named Sandeep Ramireddy. But looking at the provided context, there's no mention of Sandeep Ramireddy at all. The context is about a Finance Head named Venu GOPALA RAO KODE and details related to Form 16, tax deductions, and other financial information.

Hmm, the user might have confused the context or there's a mix-up here. The context doesn't include any personal details about Sandeep Ramireddy, like job roles, experience, skills, or achievements. Without that information, I can't create a professional summary. I need to inform the user that the necessary details aren't present in the provided context. Maybe they provided the wrong document or there's a misunderstanding. I should point out that the context is about tax forms and not a resume, so I can't help with the resume summary based on this data. It's important to clarify that to avoid incorrect information.
</think>


### Enhanced RAG Pipeline Features

In [30]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: <think>
Okay, the user is asking about Hard Negative Mining techniques. Let me check the provided context to see if there's any mention of that.

Looking through the professional summary and experience sections, I don't see any direct references to Hard Negative Mining. The context talks about machine learning projects, generative AI models, clustering, time series models, and model optimization. Techniques like feature engineering, model deployment, and using algorithms like Random Forest, K-means, ARIMA, and Prophet are mentioned. However, there's no specific mention of Hard Negative Mining, which is a technique used in training models, especially in scenarios like information retrieval or object detection, where difficult negative examples are selected to improve model performance.

Since the context doesn't include any information on Hard Negative Mining, the answer should indicate that the individual hasn't mentioned using those techniques in their experience. The response

In [36]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("professional summary of my resume of sandeep ramireddy?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'professional summary of my resume of sandeep ramireddy?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
CHENNAI, CHENNAI
03-Jun-2026
FINANCE HEADDesignation: Full Name:VENU GOPALA RAO  KODE
Notes:
1. Part B (Annexure) of the certificate in Form No.16 shall be issued by the employer.
2. If an assessee is employed under one employer during the year, Part 

'A' of the certificate in Form No.16 issued for the quarter ending on 31st March of the financial year shall contain the details
of tax deducted and deposited for all the quarters of the financial year.
3. If an assessee is employed under more than one employer during the year, each of the employers shall issue Part A of the certificate in Form No.16 pertaining to the period for which such
assessee was employed with each of the employers. Part B (Annexure) of the certificate in Form No. 16 may be issued by each of the employers or the last employer at the option of the assessee.
4. To update PAN details in Income Tax Department database, apply for 'PAN change request' through NSDL or UTITSL.
Sl. No.
Tax Deposited in respect of the

4. To update PAN details in Income Tax Department database, apply for 'PAN change request' through NSDL or UTITSL.
Sl. No.
Tax Deposited in respect of the
deductee
(Rs.)
Challan Identification Number (CIN)
BSR Code of the Bank
Branch
Date on which Tax deposi